<a href="https://colab.research.google.com/github/sidhu2690/MARL/blob/main/Hetero_GP_on_M.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
import numpy as np
import torch
import torch.nn as nn

GRID = 30
N_ROBOTS = 6
N_ADV = 1
STEPS = 500

EPSILON = 1e-6
CONTRA_THRESH = 0.2
SEED = 40

SIGMA0_SQ = 0.05
SIGMA_F = 1.0
LENGTH_SCALE = 3.0

rng = np.random.default_rng(SEED)

In [27]:
class AlphaNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

In [28]:
def make_true_map(grid):
    xs, ys = np.meshgrid(np.arange(grid), np.arange(grid))
    field = np.zeros((grid, grid))

    for _ in range(5):
        cx, cy = rng.uniform(0, grid, 2)
        amp = rng.uniform(0.5, 1.0)
        sigma = rng.uniform(1.5, 3.5)
        field += amp * np.exp(-((xs - cx) ** 2 + (ys - cy) ** 2) / (2 * sigma ** 2))

    field = (field - field.min()) / (field.max() - field.min() + 1e-8)
    return field


def random_walk(grid, steps):
    pos = rng.integers(0, grid, size=2)
    cells = []

    for _ in range(steps):
        while True:
            move = rng.choice([-1, 0, 1], size=2)
            new_pos = np.clip(pos + move, 0, grid - 1)
            if not np.array_equal(new_pos, pos):
                break

        pos = new_pos
        cells.append(tuple(pos))

    return cells


def report_value(true_val, adversarial, cell, agent_id):
    if adversarial:
        seed = abs(hash((agent_id, cell))) % (2**32)
        local_rng = np.random.default_rng(seed)
        bias_sign = 1 if (hash(cell) % 2 == 0) else -1
        bias_mag = local_rng.uniform(0.15, 0.3)
        noise = local_rng.normal(0, 0.05)
        val = true_val + bias_sign * bias_mag + noise
    else:
        val = true_val + rng.normal(0, 0.02)

    return float(np.clip(val, 0, 1))

In [29]:
def rbf_kernel(X1, X2, length_scale=1.0, sigma_f=1.0):
    sqdist = np.sum(X1**2, 1).reshape(-1, 1) + np.sum(X2**2, 1) - 2 * X1 @ X2.T
    return sigma_f ** 2 * np.exp(-0.5 / length_scale ** 2 * sqdist)


def gp_predict(X_train, y_train, X_test, noise_var, length_scale=1.0, sigma_f=1.0):
    K = rbf_kernel(X_train, X_train, length_scale, sigma_f) + np.diag(noise_var)
    K_s = rbf_kernel(X_train, X_test, length_scale, sigma_f)
    K_ss = rbf_kernel(X_test, X_test, length_scale, sigma_f) + 1e-8 * np.eye(len(X_test))

    K_inv = np.linalg.inv(K)
    mu = K_s.T @ K_inv @ y_train
    cov = K_ss - K_s.T @ K_inv @ K_s
    return mu, cov

In [30]:
net = AlphaNet()
net.load_state_dict(torch.load("alpha_net.pt", map_location="cpu"))
net.eval()

true_map = make_true_map(GRID)

is_adv = np.array([True] * N_ADV + [False] * (N_ROBOTS - N_ADV))
rng.shuffle(is_adv)

walks = [random_walk(GRID, STEPS) for _ in range(N_ROBOTS)]

grid_coords = np.array([[r, c] for r in range(GRID) for c in range(GRID)], dtype=float)

In [31]:
contradictions = np.zeros(N_ROBOTS)
D_sum = np.zeros(N_ROBOTS)
D_n = np.zeros(N_ROBOTS)
cell_data = {}

U_hist = np.zeros((STEPS, GRID, GRID))
M_hist = np.zeros((STEPS, GRID, GRID))
sd_hist = np.zeros((STEPS, GRID, GRID))
alpha_hist = np.zeros((STEPS, N_ROBOTS))

for t in range(STEPS):

    for i in range(N_ROBOTS):
        cell = walks[i][t]
        r = report_value(true_map[cell], is_adv[i], cell, i)

        others = [(j, r_j) for j, r_j in cell_data.get(cell, []) if j != i]
        if others:
            D_sum[i] += np.mean([abs(r - r_j) for _, r_j in others])
            D_n[i] += 1

            for j, r_j in others:
                if abs(r - r_j) > CONTRA_THRESH:
                    contradictions[i] += 1
                    contradictions[j] += 1

        cell_data.setdefault(cell, []).append((i, r))

    z = (contradictions - contradictions.mean()) / (contradictions.std() + EPSILON)
    D = np.divide(D_sum, D_n, out=np.zeros_like(D_sum), where=D_n > 0)
    c_norm = contradictions / (t + 1)

    feats = torch.tensor(np.stack([c_norm, z, D], axis=1), dtype=torch.float32)
    with torch.no_grad():
        alpha = net(feats).numpy()
    alpha_hist[t] = alpha

    X, M, noise_var = [], [], []
    for cell, entries in cell_data.items():
        ids = [e[0] for e in entries]
        vals = np.array([e[1] for e in entries])
        a = alpha[ids]

        u = a.sum()
        m = (a * vals).sum() / (u + EPSILON)

        U_hist[t][cell] = u
        M_hist[t][cell] = m

        X.append(cell)
        M.append(m)
        noise_var.append(SIGMA0_SQ / (u + EPSILON))

    X = np.array(X, dtype=float)
    M = np.array(M)
    noise_var = np.array(noise_var)

    mu, cov = gp_predict(
        X,
        M,
        grid_coords,
        noise_var,
        length_scale=LENGTH_SCALE,
        sigma_f=SIGMA_F,
    )

    std = np.sqrt(np.maximum(np.diag(cov), 1e-8))
    sd_hist[t] = std.reshape(GRID, GRID)


np.save("U_hist.npy", U_hist)
np.save("M_hist.npy", M_hist)
np.save("sigma_hist.npy", sd_hist)
np.save("alpha_hist.npy", alpha_hist)
np.save("is_adv.npy", is_adv)
np.save("true_map.npy", true_map)

print("saved:", U_hist.shape, M_hist.shape, sd_hist.shape)

saved: (500, 30, 30) (500, 30, 30) (500, 30, 30)


In [32]:
print(is_adv)

[False False  True False False False]


In [33]:
alpha_hist[-1]

array([1.        , 1.        , 0.01346868, 1.        , 1.        ,
       1.        ])